# Mạng nơ-ron nhiều lớp (MLP)

Bài thực hành phân loại lỗi sản phẩm trên dữ liệu tự sinh. Các mô hình được huấn luyện
từ đầu. Notebook chạy trên CPU và tự tạo dữ liệu để có thể chạy độc lập trên Kaggle.

## 1. Tạo dữ liệu lỗi sản phẩm giả lập

Mỗi hàng mô phỏng một sản phẩm. Các cột mô tả cảm biến, dây chuyền, nhà cung cấp
và ca làm việc. Nhãn `defect` được rút theo xác suất phụ thuộc vào các yếu tố đó.
Hạt giống ngẫu nhiên (random seed) giữ dữ liệu giống nhau trong cả năm notebook.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 2026
N_ROWS = 20_000
rng = np.random.default_rng(SEED)

line = rng.integers(0, 8, N_ROWS)
supplier = rng.integers(0, 16, N_ROWS)
shift = rng.choice(['day', 'swing', 'night'], N_ROWS, p=[0.45, 0.35, 0.20])
temperature = rng.normal(74, 7, N_ROWS)
pressure = rng.normal(102, 9, N_ROWS)
vibration = rng.lognormal(-0.2, 0.48, N_ROWS)
feed_speed = rng.normal(55, 8, N_ROWS)
tool_age = rng.gamma(2, 11, N_ROWS)
line_effect = rng.normal(0, 0.55, 8)
supplier_effect = rng.normal(0, 0.55, 16)

log_odds = (
    -4.65
    + 0.72 * np.log1p(vibration / 0.5)
    + 0.46 * (temperature - 74) / 7
    + 0.44 * (tool_age - 22) / 16
    + 0.36 * np.abs(pressure - 102) / 9
    + 0.18 * (feed_speed - 55) / 8
    + 0.80 * ((vibration > 1.3) & (temperature > 80))
    + 0.48 * (shift == 'night')
    + line_effect[line]
    + supplier_effect[supplier]
)
probability = 1 / (1 + np.exp(-log_odds))
frame = pd.DataFrame({
    'temperature': temperature,
    'pressure': pressure,
    'vibration': vibration,
    'feed_speed': feed_speed,
    'tool_age': tool_age,
    'production_line': [f'line_{i:02}' for i in line],
    'supplier': [f'supplier_{i:02}' for i in supplier],
    'shift': shift,
    'defect': rng.binomial(1, probability),
})
NUMERIC_COLUMNS = ['temperature', 'pressure', 'vibration', 'feed_speed', 'tool_age']
CATEGORICAL_COLUMNS = ['production_line', 'supplier', 'shift']

print(frame.head(4).round(3).to_string(index=False))
print(f"Số mẫu: {len(frame):,} | Tỷ lệ lỗi: {frame['defect'].mean():.2%}")

 temperature  pressure  vibration  feed_speed  tool_age production_line    supplier shift  defect
      73.039   109.978      1.137      55.768    50.912         line_06 supplier_13 swing       1
      67.415   110.243      0.682      54.665    12.969         line_01 supplier_14   day       0
      65.495   110.196      1.107      61.027     6.746         line_00 supplier_04   day       0
      82.278    95.993      0.968      58.438    68.024         line_05 supplier_12 night       0
Số mẫu: 20,000 | Tỷ lệ lỗi: 4.37%


## 2. Chia tập huấn luyện, xác thực và kiểm tra

Chia phân tầng (stratified split) giúp giữ tỷ lệ sản phẩm lỗi gần nhau ở ba tập.
Tập huấn luyện dùng để học mô hình và bước tiền xử lý. Tập xác thực dùng để theo dõi
chất lượng hoặc chọn trọng số. Tập kiểm tra được giữ riêng cho lần chấm cuối.

In [2]:
X = frame[NUMERIC_COLUMNS + CATEGORICAL_COLUMNS]
y = frame['defect'].to_numpy(dtype=np.int64)
X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_hold, y_hold, test_size=0.50, random_state=SEED, stratify=y_hold,
)
split_hash = hashlib.sha256(
    X_test.index.to_numpy(dtype='<i8').tobytes() + y_test.astype('<i8').tobytes()
).hexdigest()[:16]
for name, labels in [('Train', y_train), ('Validation', y_val), ('Test', y_test)]:
    print(f"{name:10s}  {len(labels):6,d} mẫu | {labels.sum():4,d} lỗi ({labels.mean():.2%})")
print(f'Mã kiểm tra tập test: {split_hash}')

Train       14,000 mẫu |  611 lỗi (4.36%)
Validation   3,000 mẫu |  131 lỗi (4.37%)
Test         3,000 mẫu |  131 lỗi (4.37%)
Mã kiểm tra tập test: 94efc544e1e83429


## Chuẩn bị đầu vào cho PyTorch

Bộ chuẩn hóa và bảng mã danh mục đều được khớp trên tập huấn luyện. Mỗi
danh mục mới hoặc bị thiếu nhận mã 0. Các tập validation và test dùng lại
các phép biến đổi này. Mô hình chạy trên CPU với tối đa hai luồng.

In [3]:
from copy import deepcopy
from time import perf_counter

import torch
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(2)
torch.manual_seed(SEED)

numeric_medians = X_train[NUMERIC_COLUMNS].median().fillna(0)
numeric_scaler = StandardScaler().fit(
    X_train[NUMERIC_COLUMNS].fillna(numeric_medians)
)

category_maps = {
    column: {
        value: index
        for index, value in enumerate(
            X_train[column].dropna().unique(), start=1
        )
    }
    for column in CATEGORICAL_COLUMNS
}
category_sizes = [
    len(category_maps[column]) + 1 for column in CATEGORICAL_COLUMNS
]


def make_tensors(frame):
    numeric = numeric_scaler.transform(
        frame[NUMERIC_COLUMNS].fillna(numeric_medians)
    ).astype(np.float32)
    categorical = np.column_stack([
        frame[column].astype(object).map(category_maps[column])
        .fillna(0).to_numpy(dtype=np.int64)
        for column in CATEGORICAL_COLUMNS
    ])
    return torch.from_numpy(numeric), torch.from_numpy(categorical)


train_inputs = make_tensors(X_train)
val_inputs = make_tensors(X_val)
test_inputs = make_tensors(X_test)
train_targets = torch.as_tensor(y_train, dtype=torch.float32)

## MLP với embedding cho biến phân loại

MLP (multilayer perceptron) nối các cột số đã chuẩn hóa với embedding,
tức vectơ học được cho từng giá trị danh mục. Hai lớp ẩn học cách phối
hợp các đặc trưng để dự đoán lỗi sản xuất.

In [4]:
class TabularMLP(nn.Module):
    def __init__(self, n_numeric, category_sizes):
        super().__init__()
        embedding_dims = [min(8, size) for size in category_sizes]
        self.embeddings = nn.ModuleList(
            nn.Embedding(size, dimension, padding_idx=0)
            for size, dimension in zip(category_sizes, embedding_dims)
        )
        self.classifier = nn.Sequential(
            nn.Linear(n_numeric + sum(embedding_dims), 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, numeric, categorical):
        embeddings = [
            embedding(categorical[:, index])
            for index, embedding in enumerate(self.embeddings)
        ]
        features = torch.cat([numeric, *embeddings], dim=1)
        return self.classifier(features).squeeze(-1)


model = TabularMLP(len(NUMERIC_COLUMNS), category_sizes)
learning_rate = 0.002
max_epochs = 12

## Huấn luyện và lưu trọng số tốt nhất

Mỗi bước học dùng `BCEWithLogitsLoss` cho bài toán phân loại nhị phân.
Sau mỗi lượt duyệt tập train, chọn trọng số có ROC-AUC cao nhất trên tập
validation. Chỉ sau khi nạp lại bộ trọng số đã chọn mới dự đoán tập test.

In [5]:
batch_size = 512
train_batches = DataLoader(
    TensorDataset(*train_inputs, train_targets),
    batch_size=batch_size,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
loss_fn = nn.BCEWithLogitsLoss()


@torch.inference_mode()
def predict_probs(inputs):
    model.eval()
    numeric, categorical = inputs
    predictions = []
    for start in range(0, len(numeric), batch_size):
        rows = slice(start, start + batch_size)
        logits = model(numeric[rows], categorical[rows])
        predictions.append(torch.sigmoid(logits))
    return torch.cat(predictions).numpy()


best_auc = -np.inf
fit_started = perf_counter()
for epoch in range(1, max_epochs + 1):
    model.train()
    total_loss = 0.0
    for numeric, categorical, target in train_batches:
        optimizer.zero_grad(set_to_none=True)
        logits = model(numeric, categorical)
        loss = loss_fn(logits, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(target)

    validation_auc = roc_auc_score(y_val, predict_probs(val_inputs))
    if validation_auc > best_auc:
        best_auc = validation_auc
        best_epoch = epoch
        best_weights = deepcopy(model.state_dict())

    mean_loss = total_loss / len(train_targets)
    print(f"Epoch {epoch:02d} | Train loss {mean_loss:.4f} | Val ROC-AUC {validation_auc:.4f}")

model.load_state_dict(best_weights)
fit_seconds = perf_counter() - fit_started
val_probs = predict_probs(val_inputs)
test_probs = predict_probs(test_inputs)
print(f"Epoch đã chọn {best_epoch:02d} | Val ROC-AUC {best_auc:.4f}")

Epoch 01 | Train loss 0.3308 | Val ROC-AUC 0.5187
Epoch 02 | Train loss 0.1828 | Val ROC-AUC 0.6867
Epoch 03 | Train loss 0.1676 | Val ROC-AUC 0.7180
Epoch 04 | Train loss 0.1629 | Val ROC-AUC 0.7352
Epoch 05 | Train loss 0.1594 | Val ROC-AUC 0.7470
Epoch 06 | Train loss 0.1572 | Val ROC-AUC 0.7524
Epoch 07 | Train loss 0.1563 | Val ROC-AUC 0.7568
Epoch 08 | Train loss 0.1556 | Val ROC-AUC 0.7581
Epoch 09 | Train loss 0.1540 | Val ROC-AUC 0.7594
Epoch 10 | Train loss 0.1547 | Val ROC-AUC 0.7591
Epoch 11 | Train loss 0.1546 | Val ROC-AUC 0.7602
Epoch 12 | Train loss 0.1534 | Val ROC-AUC 0.7593
Epoch đã chọn 11 | Val ROC-AUC 0.7602


## Đánh giá và đọc kết quả

ROC AUC đo khả năng xếp sản phẩm lỗi cao hơn sản phẩm đạt. Average Precision
(AP) tóm tắt độ chính xác của danh sách cảnh báo khi tăng số mẫu được phát hiện.
Tỷ lệ lỗi của tập test là mốc AP để tham khảo với cách xếp hạng ngẫu nhiên.

In [6]:
MODEL_SLUG = 'mlp'
for name, labels, scores in [('validation', y_val, val_probs), ('test', y_test, test_probs)]:
    assert scores.shape == labels.shape, f'Sai kích thước dự đoán trên {name}'
    assert np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all()

metrics = {
    'model': MODEL_SLUG,
    'seed': SEED,
    'train_rows': len(y_train),
    'validation_rows': len(y_val),
    'test_rows': len(y_test),
    'test_positive_count': int(y_test.sum()),
    'test_positive_rate': float(y_test.mean()),
    'test_split_hash': split_hash,
    'validation_roc_auc': float(roc_auc_score(y_val, val_probs)),
    'test_roc_auc': float(roc_auc_score(y_test, test_probs)),
    'test_average_precision': float(average_precision_score(y_test, test_probs)),
    'fit_seconds': float(fit_seconds),
}
on_kaggle = Path('/kaggle/working').is_dir()
output_dir = Path('/kaggle/working') if on_kaggle else Path('results')
output_dir.mkdir(exist_ok=True)
output_file = output_dir / f'{MODEL_SLUG}-metrics.json'
output_file.write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')
print(f"ROC AUC validation: {metrics['validation_roc_auc']:.4f}")
print(f"ROC AUC test:       {metrics['test_roc_auc']:.4f}")
print(f"Average Precision:  {metrics['test_average_precision']:.4f}")
print(f"Tỷ lệ lỗi tập test: {metrics['test_positive_rate']:.2%}")
print(f"Thời gian fit:      {metrics['fit_seconds']:.2f} giây")
print(f"Đã lưu kết quả:    {output_file}")

ROC AUC validation: 0.7602
ROC AUC test:       0.7704
Average Precision:  0.1623
Tỷ lệ lỗi tập test: 4.37%
Thời gian fit:      1.34 giây
Đã lưu kết quả:    results\mlp-metrics.json
